# Inspect + Validate Retweet Graph
Single notebook for quick schema checks and sanity stats.

In [ ]:
import torch
from pprint import pprint

graph_path = 'data/data/midterm/graphs/retweet_graph.pt'
raw = torch.load(graph_path, map_location='cpu')
print('Loaded:', graph_path)
print('Keys:')
pprint(sorted(raw.keys()))

In [ ]:
required = ['x', 'edge_index', 'user_ids', 'feature_names', 'y', 'label_names']
missing = [k for k in required if k not in raw]
assert not missing, f'Missing required keys: {missing}'

x = raw['x']
ei = raw['edge_index']
y = raw['y']

assert x.dim() == 2
assert ei.dim() == 2 and ei.shape[0] == 2
assert y.dim() == 1
assert len(raw['user_ids']) == x.shape[0]
assert len(raw['feature_names']) == x.shape[1]
assert y.shape[0] == x.shape[0]

if ei.numel() > 0:
    assert int(ei.min()) >= 0
    assert int(ei.max()) < x.shape[0]

assert not torch.isnan(x).any()
ea = raw.get('edge_attr')
if ea is not None:
    assert ea.shape[0] == ei.shape[1]
    assert not torch.isnan(ea).any()

print('Schema checks: PASS')

In [ ]:
print('x:', tuple(raw['x'].shape), raw['x'].dtype)
print('edge_index:', tuple(raw['edge_index'].shape), raw['edge_index'].dtype)
if raw.get('edge_attr') is not None:
    print('edge_attr:', tuple(raw['edge_attr'].shape), raw['edge_attr'].dtype)
print('feature_names:', len(raw.get('feature_names', [])))
print('edge_attr_feature_names:', raw.get('edge_attr_feature_names', []))
print('labeled nodes:', int((raw['y'] >= 0).sum().item()), '/', raw['y'].shape[0])

print('edge_index_views:', list(raw.get('edge_index_views', {}).keys()))
print('edge_attr_views:', list(raw.get('edge_attr_views', {}).keys()))
print('target_edge_index_views:', list(raw.get('target_edge_index_views', {}).keys()))